# 04 문제 2 (공공데이터) - Titanic 생존 예측 [정답]

> **공공데이터**: 타이타닉 탑승객 데이터. 로그인/키 없이 URL로 바로 로드.
> 출처: https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
> **목표**: 탑승객 정보로 **생존 여부(Survived: 0=사망, 1=생존)** 를 맞히는 이진분류.
> 04에서 배운 워크플로우를 새 데이터에 그대로 적용해본다.

### 1단계. 공공데이터 로드 (URL, 로그인 불필요)

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

### 2단계. EDA - 타겟/결측치 확인

In [ ]:
print(df['Survived'].value_counts())        # 0 사망 / 1 생존
df[['Age', 'Embarked']].isnull().sum()      # Age/Embarked 결측

### 3단계. 특성 선택 & 전처리
이름/티켓번호처럼 예측에 바로 쓰기 어려운 컬럼은 제외하고, 핵심 특성만 사용.

In [ ]:
cols = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
df = df[cols].copy()

df['Age'] = df['Age'].fillna(df['Age'].median())          # 나이 결측 -> 중앙값
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

y = df['Survived']
X = pd.get_dummies(df.drop(columns=['Survived']))          # Sex, Embarked 원핫
print('X:', X.shape)

### 4단계. 분할 & 학습

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
rf = RandomForestClassifier(random_state=0)
rf.fit(X_tr, y_tr)

### 5단계. 평가

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

pred = rf.predict(X_val)
proba = rf.predict_proba(X_val)[:, 1]
print('정확도 :', round(accuracy_score(y_val, pred), 4))
print('ROC-AUC:', round(roc_auc_score(y_val, proba), 4))
print()
print(classification_report(y_val, pred))

### 6단계. 중요 특성 확인 (누가 살았나?)

In [ ]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(6)
# 보통 Sex(성별), Fare(요금), Age(나이), Pclass(객실등급)가 상위 -> 여성/고급객실 생존율 높음

## 정리
- 04 워크플로우(로드->EDA->전처리->분할->학습->평가)를 **새 공공데이터에 그대로** 적용.
- 정확도 ~0.79, ROC-AUC ~0.82.
- 중요 특성: 성별/요금/나이/객실등급 -> '여성과 상류층의 생존율이 높았다'는 역사적 사실과 일치.